# Lab 00 — 把一條線接起來

這一節把五個環節接成一條線：作業系統的 counter、node_exporter、Prometheus、一支自己寫的偵測
服務，以及 Grafana 與告警。接完之後它會一直跑下去，不需要有人按執行。

![Lab 00 的資料流](diagrams/lab00_pipeline.svg)

這門課只寫其中一個方框，`detector.py`。它做三件事：向 Prometheus 查一段流量、算出一個偏離分數、
把分數曝露成 `/metrics`。分數被 Prometheus 抓回去之後就是一般指標，Grafana 查得到，告警規則
也用得上，沒有人需要知道它是 Python 算的。

後面兩節動的都是同一個地方。Lab 01 換掉算基線的方法，Lab 02 在分數外面包門檻與政策。管線在
這一節接好，之後不再碰它。

## 1. 三個服務要先在跑

| 元件 | 位址 | 這一節用它做什麼 |
| --- | --- | --- |
| Prometheus | <http://localhost:9090> | 存指標、被 detector 查詢、評估告警規則 |
| node_exporter | <http://localhost:9100/metrics> | 曝露這台機器的網路 counter |
| Grafana | <http://localhost:3000> | 把原始速率與分數畫在同一張圖上 |

安裝與啟動見 [`labs/getting-started/`](../getting-started/README.md)。Windows 的 exporter 是
`windows_exporter`，聽在 9182。第四個服務 `detector.py` 在第 4 節才啟動，現在還不用管它。

In [ ]:
import pathlib
import sys
import time
import urllib.request
from collections import deque

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

PROJECT_ROOT = pathlib.Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "environments").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
LAB_DIR = PROJECT_ROOT / "labs" / "workshop"
sys.path.insert(0, str(LAB_DIR))

# 這一節不自己寫演算法，直接用 detector.py 裡的那一個。notebook 與常駐服務跑的是同一段程式，
# 所以在這裡看到的分數，就是 Prometheus 待會抓回去的那個分數。
from detector import EXPORT_PORT, PROMETHEUS_URL, pick_target, promql, rolling_zscore

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Georgia", "Times New Roman", "DejaVu Serif"],
    "figure.dpi": 110, "axes.titlesize": 11, "axes.titlelocation": "left",
    "axes.labelsize": 9, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
})
pd.set_option("display.width", 120)


def reachable(url, timeout=2.0):
    """能不能拿到 HTTP 回應。用 stdlib，因為這一格要在任何環境都跑得起來。"""
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            return response.status < 400
    except Exception:
        return False


SERVICES = [
    ("Prometheus", "http://localhost:9090/-/ready", "第 2 節之後每一格都要用"),
    ("node_exporter", "http://localhost:9100/metrics", "Windows 是 9182"),
    ("Grafana", "http://localhost:3000/api/health", "第 6 節才用得到"),
]

print(f"專案根目錄  {PROJECT_ROOT}")
print()
for name, url, note in SERVICES:
    print(f"  {'OK  ' if reachable(url) else '沒回應'}  {name:<16}{url:<42}{note}")

## 2. 原始訊號：counter 與 rate

`node_network_receive_bytes_total` 是 counter，從開機起只增不減，直到重開機歸零。直接畫它會得到
一條往上的斜線，看不出任何事情。有意義的是它的斜率，也就是每秒多少 bytes，PromQL 用 `rate()`
取斜率。

```promql
node_network_receive_bytes_total{device="en0"}             # 累積量
rate(node_network_receive_bytes_total{device="en0"}[1m])   # 每秒速率
```

下面這一格先確認 Prometheus 真的抓到 exporter，再挑出哪一張網卡在傳資料。虛擬介面很多，挑錯
的話後面所有的圖都是一條平的零線。`up` 值得記住：它是 Prometheus 為每一個 target 自己產生的
指標，抓得到是 1，抓不到是 0，空白的畫面該查的第一條 PromQL 就是它。

In [ ]:
targets = requests.get(f"{PROMETHEUS_URL}/api/v1/targets", timeout=5).json()
print("Prometheus 現在在抓的 target")
for t in targets["data"]["activeTargets"]:
    print(f"  {t['labels']['job']:<18}{t['scrapeUrl']:<38}{t['health']}")

# detector.py 用哪一張網卡，這裡就用哪一張，兩邊不會各挑各的。
METRIC, LABEL, IFACE = pick_target()
print(f"\n選中的網卡  {IFACE}   指標  {METRIC}")

print("\ncounter 與 rate 的差別，同一個瞬間")
raw = promql(f'{METRIC}{{{LABEL}="{IFACE}"}}')
per_s = promql(f'rate({METRIC}{{{LABEL}="{IFACE}"}}[1m])')
print(f"  累積量  {float(raw[0]['value'][1]):>18,.0f} bytes    開機以來的總和")
print(f"  rate    {float(per_s[0]['value'][1]):>18,.0f} bytes/s  這才是流量")

## 3. 訊號處理：先在 notebook 裡算一次

偵測要處理的不是單一個值，是一段歷史。下面把最近一小時的速率拉回來，對每一個點問同一個問題：
它離前面那段時間的常態有多遠。答案用標準差當單位，這就是 z 分數。

算法本身只有三行，重點在它吃的是什麼。分數用「這個值進入視窗之前」的視窗算，否則異常值會先
被算進平均與標準差裡，自己抬高自己的基線，於是越大的異常越不容易被抓到。下面的迴圈刻意寫成
跟 `detector.py` 的主迴圈同一個順序。

In [ ]:
def promql_range(query, minutes=60, step="15s"):
    """拉一段時間序列回來。Grafana 畫折線圖的時候打的就是這個端點。"""
    end = time.time()
    response = requests.get(f"{PROMETHEUS_URL}/api/v1/query_range",
                            params={"query": query, "start": end - minutes * 60,
                                    "end": end, "step": step}, timeout=15)
    result = response.json()["data"]["result"]
    if not result:
        raise SystemExit("這段時間沒有資料。Prometheus 才剛啟動的話，等幾分鐘再執行一次。")
    pairs = result[0]["values"]
    return pd.DataFrame({
        "timestamp": pd.to_datetime([float(t) for t, _ in pairs], unit="s"),
        "rate_bps": [float(v) for _, v in pairs],
    })

history = promql_range(f'rate({METRIC}{{{LABEL}="{IFACE}"}}[1m])')

# 跟 detector.py 主迴圈同樣的順序：先用舊視窗評分，再把新值放進視窗。
WINDOW = 40
window, scores = deque(maxlen=WINDOW), []
for value in history["rate_bps"]:
    scores.append(rolling_zscore(window, value))
    window.append(value)
history["score"] = scores

print(f"{len(history)} 個樣本，{history['timestamp'].min()} 到 {history['timestamp'].max()}")
print(f"速率中位數  {history['rate_bps'].median():>12,.0f} bytes/s")
print(f"分數絕對值的 95 百分位  {history['score'].abs().quantile(0.95):.2f}")
print(f"超過 3 的樣本  {int((history['score'].abs() > 3).sum())} 個")

fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 5), sharex=True,
                                  gridspec_kw={"height_ratios": [2, 1]})
top.plot(history["timestamp"], history["rate_bps"], color="#3B7DD8", lw=1.2)
top.set_title(f"receive rate on {IFACE}", loc="left")
top.set_ylabel("bytes/s")
bottom.plot(history["timestamp"], history["score"], color="#E0752D", lw=1.2)
bottom.axhline(3, color="#D6455D", ls="--", lw=1.0)
bottom.axhline(-3, color="#D6455D", ls="--", lw=1.0)
bottom.set_title("rolling z-score, threshold at 3", loc="left")
bottom.set_ylabel("z")
fig.tight_layout()
plt.show()

下面那格的分數幾乎貼著 0，代表這段時間沒有事情發生，這是對的。想看它動起來，就下載一個大檔案
再重新執行這一格。

看得出這條線的性格之後有一件事值得先想：視窗是 40 個樣本、每 15 秒一個，也就是大約 10 分鐘。
視窗拉長，基線更穩，但是慢慢爬上去的異常會被基線跟著吃掉；視窗縮短，反應快，但是平常的起伏
就足以越線。這個取捨在 Lab 01 用真的故障資料量化。

## 4. 同一段程式，變成常駐服務

上面那一格算完就結束了。要讓它一直算下去，需要的只是一個迴圈，加上一個讓 Prometheus 抓得到
的 HTTP 端點。`prometheus_client` 這個官方套件把後者變成一行 `start_http_server`。

[`detector.py`](detector.py) 就是這樣一支程式，剛才 `import` 進來的 `rolling_zscore` 正是它每
15 秒會呼叫一次的那個函式。**另外開一個終端機**，在 repo 根目錄執行，這個視窗要一直留著：

```bash
python labs/workshop/detector.py
```

看到 `detector 監看 ...` 之後回來執行下一格。

In [ ]:
metrics = requests.get(f"http://localhost:{EXPORT_PORT}/metrics", timeout=5).text
print(f"detector 在 :{EXPORT_PORT} 曝露的東西\n")
for line in metrics.splitlines():
    if line.startswith("aiops_") and not line.startswith("#"):
        print(f"  {line}")

分數是 0 而樣本數還很少是正常的，視窗要幾分鐘才裝得滿。

值得停下來看一眼的是這幾行的形狀：`aiops_traffic_score{device="en0"} 0.0`，跟 node_exporter 吐出
來的每一行長得一模一樣。指標名、label、值，Prometheus 認得的就是這三樣東西。自己寫的服務與
官方 exporter 在這一層沒有分別，這是整條線接得起來的原因。

## 5. 讓 Prometheus 抓分數

三份 `infra/prometheus/prometheus.*.yml` 裡已經寫好 `aiops-detector` 這個 job，指向
`localhost:9200`，不用自己新增。如果 Prometheus 是在 detector 之前啟動的，讓它重新讀一次設定：

```bash
curl -X POST http://localhost:9090/-/reload
```

Windows PowerShell 是 `Invoke-WebRequest -Method Post http://localhost:9090/-/reload`。回 `405`
表示啟動時沒有帶 `--web.enable-lifecycle`，直接重啟 Prometheus 也可以。

In [ ]:
print("三個 job 的 up")
for row in promql("up"):
    print(f"  {row['metric']['job']:<18}{row['value'][1]}")

print("\n分數從 Prometheus 這一側查回來")
for row in promql("aiops_traffic_score"):
    print(f"  device={row['metric'].get('device', '?'):<10}{float(row['value'][1]):>8.2f}")

# 繞一圈之後，PromQL 對它做得了跟對任何指標一樣的事。
print("\n最近 10 分鐘的分數峰值")
for row in promql("max_over_time(abs(aiops_traffic_score)[10m:15s])"):
    print(f"  device={row['metric'].get('device', '?'):<10}{float(row['value'][1]):>8.2f}")

`up{job="aiops-detector"}` 是 1，而且 `aiops_traffic_score` 查得回來，這條線就閉合了：訊號從網卡
出發，經過 exporter 與 Prometheus 到 Python，算完再回到 Prometheus。

最後那一條 `max_over_time(abs(...)[10m:15s])` 值得看一眼。它是 subquery，把分數當成一般的時間
序列去取區間最大值。分數送回 Prometheus 換到的就是這個：所有 PromQL 的工具，對它一律有效。

## 6. Grafana 上的三張 panel

到 <http://localhost:3000> 建一張 dashboard，三張 panel 分別畫原始速率、分數、告警狀態。逐步的
建法在 [`dashboard.md`](dashboard.md)，跟著建完再回來。

三張都用 Prometheus 這一個 datasource。這門課沒有第二個資料來源，因為分數已經在 Prometheus
裡面了。

## 7. 告警：讓它真的響

`infra/prometheus/alerts.yml` 裡的 `TrafficAnomaly` 打在 `aiops_traffic_score` 上，門檻 3，
`for: 1m`。

`for` 是這條規則裡最該理解的一個字。條件成立的瞬間告警不會送出去，它先進入 Pending，要連續
成立滿一分鐘才轉成 Firing。這是時間閘，作用是擋掉單一個越線的取樣。同一件事 Lab 02 在 Python
那一側寫成「連續 n 個樣本才算數」，兩邊是同一個想法的兩種寫法。

現在讓它響：下載一個大檔案，或執行 `curl -o /dev/null https://speed.hetzner.de/1GB.bin`，然後
在 <http://localhost:9090/alerts> 看著它從 Normal 走到 Pending 再到 Firing。下面這一格印的是同
一份狀態。

In [ ]:
groups = requests.get(f"{PROMETHEUS_URL}/api/v1/rules", timeout=5).json()["data"]["groups"]
print(f"{'規則':<22}{'型別':<12}{'狀態':<12}{'for'}")
for group in groups:
    for rule in group["rules"]:
        name = rule.get("name", "?")
        kind = rule["type"]
        state = rule.get("state", "-")
        hold = f"{rule['duration']:.0f}s" if rule.get("duration") else "-"
        print(f"  {name:<22}{kind:<12}{state:<12}{hold}")

firing = [a for g in groups for r in g["rules"] for a in r.get("alerts", [])]
print(f"\n現在有 {len(firing)} 則告警處於 Pending 或 Firing")
for alert in firing:
    print(f"  {alert['labels']['alertname']:<20}{alert['state']:<10}{alert['annotations'].get('summary', '')}")

## 8. 四種壞法

一條線有幾個接點就有幾種壞法。四種各弄一次，每一次先預測畫面上會看到什麼，再動手。

1. **關掉 detector。** 在那個終端機按 `Ctrl+C`，等一分鐘。`up{job="aiops-detector"}` 變成 0，
   `DetectorDown` 開始響。分數那張 panel 斷線，但是 `TrafficAnomaly` 不會響，因為它的條件
   再也不會成立。監看系統自己掛掉的時候畫面看起來是安靜的，這就是 `DetectorDown` 存在的理由。
2. **關掉 node_exporter。** detector 查不到資料，分數停在最後一個值，速率那張 panel 也斷線。
   `up{job="node-exporter"}` 變成 0。
3. **時間範圍選錯。** 把 Grafana 的時間選擇器改成 Last 5 minutes 再改成 Last 2 days，panel 變
   空白，而且沒有任何錯誤訊息。
4. **規則檔改壞。** 在 `alerts.yml` 裡把某個指標名打錯再 reload，規則載得進去，只是永遠不會
   成立。`promtool check rules infra/prometheus/alerts.yml` 檢查得出語法錯誤，檢查不出指標名
   打錯。

第三種與第四種跟前兩種的差別在痕跡。前兩種留下 `up` 的狀態變化，後兩種什麼都不留。所以看到
空白畫面的時候順序是固定的：先查 `up`，再看時間範圍，最後才懷疑資料本身。

## 9. 離開之前

下面幾題自己回答一次，答不出來的那一題就是這一節還沒接上的地方。

1. counter 與 rate 的差別，以及為什麼直接畫 `node_network_receive_bytes_total` 沒有意義。
2. `aiops_traffic_score` 是怎麼進到 Prometheus 的，中間經過哪些元件。答案裡不應該出現「匯入」
   或「上傳」。
3. `TrafficAnomaly` 這條規則怎麼知道分數是 Python 算的。它不知道，說明為什麼這件事是優點。
4. `for: 1m` 擋掉的是什麼，不換這個參數的話還有什麼方法達到同樣的效果。
5. detector 掛掉的時候，哪一條規則會響，哪一條不會，為什麼。

管線到這裡就完整了。Lab 01 與 Lab 02 從頭到尾只做一件事：把 `rolling_zscore` 換成撐得住真實
流量的版本。下一節先問一個更前面的問題，同一段流量換一種基線就換一種判定，那應該換哪一種。